In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {
    "numpy":"numpy",
    "pandas":"pandas",
    "matplotlib":"matplotlib",
    "sklearn":"scikit-learn",
    "statsmodels":"statsmodels",
    "pulp":"pulp",
    "tensorflow":"tensorflow",
}

installed=[]

for module,package in REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        print("Instalando",package)
        subprocess.check_call([
            sys.executable,"-m","pip","install",package
        ])
        installed.append(package)

if installed:
    print("Instalados:",installed)
    print("Se necessário, reinicie o kernel uma vez.")
else:
    print("Dependências: OK")


In [ ]:
from pathlib import Path
import json, hashlib, random, warnings
import os, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX

import pulp
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")

SEED=123
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

def find_repo_root(start=None):
    """Locate the FAME repository root from the current working directory."""
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
            and (candidate / "results").exists()
        ):
            return candidate
    raise RuntimeError(
        "FAME repository root not found. Run this notebook from inside a clone "
        "of the FAME repository."
    )

REPO_ROOT = find_repo_root()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
REPRODUCED_DIR = REPO_ROOT / "reproduced"
REPRODUCED_DIR.mkdir(parents=True, exist_ok=True)

OUT = REPRODUCED_DIR / "energy" / "temporal_replication"
OUT.mkdir(parents=True, exist_ok=True)

_cbc_env = os.environ.get("FAME_CBC_EXE")
_cbc_cmd = _cbc_env or shutil.which("cbc")
CBC_EXE = Path(_cbc_cmd).expanduser() if _cbc_cmd else None

BASE_VOLL=10_000.0
EMERGENCY_COST_MULTIPLIER=10.0

CAPACITY_MARGINS=np.array([1.00,1.10,1.20,1.30,1.40],dtype=float)
VOLL_MULTIPLIERS=np.array([0.5,1.0,2.0,5.0],dtype=float)

THETA_GRID=np.round(
    np.arange(-0.30,0.3001,0.025),
    3
)

print("Repository:", REPO_ROOT)
print("Output:", OUT)
print("CBC executable:", CBC_EXE if CBC_EXE else "PuLP/default solver discovery")
print("Theta grid:",THETA_GRID.tolist())
print("Margins:",CAPACITY_MARGINS.tolist())
print("VOLL multipliers:",VOLL_MULTIPLIERS.tolist())


In [ ]:
TRANSNETBW_OVERRIDE = os.environ.get("FAME_TRANSNETBW_DATA")
DEFAULT_TRANSNETBW = (
    RAW_DATA_DIR
    / "energy"
    / "transnetbw_actual_forecast_hourly_utc_2015_2025.csv"
)
canonical = (
    Path(TRANSNETBW_OVERRIDE).expanduser()
    if TRANSNETBW_OVERRIDE
    else DEFAULT_TRANSNETBW
)

if not canonical.exists():
    raise FileNotFoundError(
        "TransnetBW input file not found.\n"
        f"Expected: {DEFAULT_TRANSNETBW}\n"
        "Alternatively set FAME_TRANSNETBW_DATA to the file path.\n"
        "See data/SOURCES_AND_RECONSTRUCTION.md."
    )

z=pd.read_csv(canonical)
z["timestamp_utc"]=pd.to_datetime(z["timestamp_utc"],utc=True)

required={"timestamp_utc","actual_load_mw","dayahead_forecast_mw"}
missing=required.difference(z.columns)
if missing:
    raise RuntimeError(f"Colunas ausentes: {sorted(missing)}")

z["date"]=z["timestamp_utc"].dt.floor("D")

daily=(
    z.groupby("date",as_index=False)
     .agg(
         actual_peak=("actual_load_mw","max"),
         entsoe_forecast_peak=("dayahead_forecast_mw","max"),
         n_hours=("timestamp_utc","size"),
     )
     .sort_values("date")
     .reset_index(drop=True)
)

daily=daily[daily["n_hours"]>=23].copy()

daily["year"]=daily["date"].dt.year
daily["dow"]=daily["date"].dt.dayofweek
daily["dow_sin"]=np.sin(2*np.pi*daily["dow"]/7)
daily["dow_cos"]=np.cos(2*np.pi*daily["dow"]/7)
daily["doy"]=daily["date"].dt.dayofyear
daily["doy_sin"]=np.sin(2*np.pi*daily["doy"]/366)
daily["doy_cos"]=np.cos(2*np.pi*daily["doy"]/366)

print("Arquivo:",canonical)
print("Anos:",sorted(daily.year.unique()))
print("Dias:",len(daily))


In [ ]:
replications=[]

for test_year in range(2017,2026):
    cal_year=test_year-1
    dev_years=list(range(2015,cal_year))

    replications.append({
        "replication_id":f"E{test_year}",
        "development_start":min(dev_years),
        "development_end":max(dev_years),
        "calibration_year":cal_year,
        "test_year":test_year,
    })

replications=pd.DataFrame(replications)

needed=set(range(2015,2026))
available=set(daily.year.unique())

if not needed.issubset(available):
    raise RuntimeError(
        f"Anos ausentes: {sorted(needed.difference(available))}"
    )

display(replications)


In [ ]:
EXOG_COLS=["dow_sin","dow_cos","doy_sin","doy_cos"]
CAL_COLS=EXOG_COLS

def regression_metrics(y_true,y_pred):
    y_true=np.asarray(y_true,dtype=float)
    y_pred=np.asarray(y_pred,dtype=float)

    return {
        "rmse":float(np.sqrt(mean_squared_error(y_true,y_pred))),
        "mae":float(mean_absolute_error(y_true,y_pred)),
        "mape":float(np.mean(
            np.abs(y_true-y_pred)/np.clip(np.abs(y_true),1e-9,None)
        )),
        "bias":float(np.mean(y_pred-y_true)),
    }

SARIMAX_CANDIDATES=[
    {"order":(1,0,0),"seasonal_order":(0,0,0,7)},
    {"order":(1,0,1),"seasonal_order":(0,0,0,7)},
    {"order":(2,0,0),"seasonal_order":(0,0,0,7)},
    {"order":(1,0,0),"seasonal_order":(1,0,0,7)},
    {"order":(1,0,1),"seasonal_order":(1,0,0,7)},
]

def select_sarimax(dev):
    dev=dev.sort_values("date").reset_index(drop=True)
    split=max(60,int(len(dev)*0.80))
    split=min(split,len(dev)-30)

    tr=dev.iloc[:split]
    va=dev.iloc[split:]

    rows=[]

    for cfg in SARIMAX_CANDIDATES:
        try:
            fit=SARIMAX(
                tr["actual_peak"].values,
                exog=tr[EXOG_COLS].values,
                order=cfg["order"],
                seasonal_order=cfg["seasonal_order"],
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(disp=False,maxiter=150)

            pred=fit.get_forecast(
                steps=len(va),
                exog=va[EXOG_COLS].values
            ).predicted_mean

            met=regression_metrics(va["actual_peak"],pred)

            rows.append({
                **cfg,
                **met,
                "aic":float(fit.aic)
            })

        except Exception:
            rows.append({
                **cfg,
                "rmse":np.inf,
                "mae":np.inf,
                "mape":np.inf,
                "bias":np.nan,
                "aic":np.inf
            })

    tab=pd.DataFrame(rows)
    ok=tab[np.isfinite(tab.rmse)]

    if ok.empty:
        raise RuntimeError("Nenhum SARIMAX convergiu.")

    best=ok.sort_values(["rmse","mae","aic"]).iloc[0]

    return tuple(best["order"]),tuple(best["seasonal_order"]),tab

def sarimax_sequential_forecast(dev,future,order,seasonal_order):
    fit=SARIMAX(
        dev["actual_peak"].values,
        exog=dev[EXOG_COLS].values,
        order=order,
        seasonal_order=seasonal_order,
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False,maxiter=200)

    preds=[]

    for row in future.sort_values("date").itertuples(index=False):
        ex=np.array([[getattr(row,c) for c in EXOG_COLS]],dtype=float)

        pred=float(
            fit.get_forecast(
                steps=1,
                exog=ex
            ).predicted_mean[0]
        )

        preds.append(pred)

        fit=fit.append(
            endog=[float(row.actual_peak)],
            exog=ex,
            refit=False
        )

    return np.asarray(preds)


In [ ]:
LSTM_CANDIDATES=[
    {"lookback":7,"units":16,"lr":1e-3},
    {"lookback":14,"units":16,"lr":1e-3},
    {"lookback":14,"units":32,"lr":1e-3},
    {"lookback":21,"units":32,"lr":5e-4},
]

def make_sequences(df,lookback):
    df=df.sort_values("date").reset_index(drop=True)
    vals=df["actual_peak"].values.astype(float)

    Xs,Xc,y=[],[],[]

    for i in range(lookback,len(df)):
        Xs.append(vals[i-lookback:i].reshape(-1,1))
        Xc.append(df.loc[i,CAL_COLS].values.astype(float))
        y.append(vals[i])

    return np.asarray(Xs),np.asarray(Xc),np.asarray(y)

def build_lstm(lookback,units,lr):
    seq_in=Input(shape=(lookback,1))
    cal_in=Input(shape=(len(CAL_COLS),))

    x=LSTM(units)(seq_in)
    x=Concatenate()([x,cal_in])
    x=Dense(16,activation="relu")(x)
    out=Dense(1)(x)

    model=Model([seq_in,cal_in],out)
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="mse"
    )

    return model

def select_lstm(dev):
    rows=[]

    for cfg in LSTM_CANDIDATES:
        Xs,Xc,y=make_sequences(
            dev,
            int(cfg["lookback"])
        )

        split=max(30,int(len(y)*0.80))
        split=min(split,len(y)-20)

        if split<=0 or len(y)-split<10:
            continue

        Xs_tr,Xs_va=Xs[:split],Xs[split:]
        Xc_tr,Xc_va=Xc[:split],Xc[split:]
        y_tr,y_va=y[:split],y[split:]

        scaler=StandardScaler().fit(
            y_tr.reshape(-1,1)
        )

        mu,sd=scaler.mean_[0],scaler.scale_[0]

        Xs_tr_sc=(Xs_tr-mu)/sd
        Xs_va_sc=(Xs_va-mu)/sd

        y_tr_sc=scaler.transform(
            y_tr.reshape(-1,1)
        ).ravel()

        y_va_sc=scaler.transform(
            y_va.reshape(-1,1)
        ).ravel()

        tf.keras.backend.clear_session()
        tf.random.set_seed(SEED)

        model=build_lstm(
            int(cfg["lookback"]),
            int(cfg["units"]),
            float(cfg["lr"])
        )

        es=EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True
        )

        hist=model.fit(
            [Xs_tr_sc,Xc_tr],
            y_tr_sc,
            validation_data=([Xs_va_sc,Xc_va],y_va_sc),
            epochs=200,
            batch_size=32,
            verbose=0,
            callbacks=[es]
        )

        pred_sc=model.predict(
            [Xs_va_sc,Xc_va],
            verbose=0
        ).ravel()

        pred=scaler.inverse_transform(
            pred_sc.reshape(-1,1)
        ).ravel()

        rows.append({
            **cfg,
            **regression_metrics(y_va,pred),
            "epochs":len(hist.history["loss"])
        })

    tab=pd.DataFrame(rows)

    if tab.empty:
        raise RuntimeError("Nenhuma LSTM avaliada.")

    best=tab.sort_values(["rmse","mae"]).iloc[0]

    return {
        "lookback":int(best["lookback"]),
        "units":int(best["units"]),
        "lr":float(best["lr"])
    },tab

def fit_lstm_final(dev,cfg):
    Xs,Xc,y=make_sequences(
        dev,
        cfg["lookback"]
    )

    scaler=StandardScaler().fit(
        y.reshape(-1,1)
    )

    mu,sd=scaler.mean_[0],scaler.scale_[0]

    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model=build_lstm(
        cfg["lookback"],
        cfg["units"],
        cfg["lr"]
    )

    es=EarlyStopping(
        monitor="loss",
        patience=20,
        restore_best_weights=True
    )

    model.fit(
        [(Xs-mu)/sd,Xc],
        scaler.transform(y.reshape(-1,1)).ravel(),
        epochs=250,
        batch_size=32,
        verbose=0,
        callbacks=[es]
    )

    return model,scaler

def lstm_sequential_forecast(
    model,
    scaler,
    history,
    future,
    lookback
):
    observed=history.sort_values("date")[
        "actual_peak"
    ].astype(float).tolist()

    mu,sd=scaler.mean_[0],scaler.scale_[0]
    preds=[]

    for row in future.sort_values("date").itertuples(index=False):
        seq=np.asarray(
            observed[-lookback:],
            dtype=float
        ).reshape(1,lookback,1)

        cal=np.array(
            [[getattr(row,c) for c in CAL_COLS]],
            dtype=float
        )

        pred_sc=model.predict(
            [(seq-mu)/sd,cal],
            verbose=0
        ).ravel()[0]

        pred=float(
            scaler.inverse_transform(
                np.array([[pred_sc]])
            )[0,0]
        )

        preds.append(pred)
        observed.append(float(row.actual_peak))

    return np.asarray(preds)


In [ ]:
RTS_GEN_OVERRIDE = os.environ.get("FAME_RTS_GEN_DATA")
DEFAULT_GEN_FILE = (
    RAW_DATA_DIR
    / "energy"
    / "RTS-GMLC"
    / "RTS_Data"
    / "SourceData"
    / "gen.csv"
)
GEN_FILE = (
    Path(RTS_GEN_OVERRIDE).expanduser()
    if RTS_GEN_OVERRIDE
    else DEFAULT_GEN_FILE
)

if not GEN_FILE.exists():
    raise FileNotFoundError(
        "RTS-GMLC SourceData/gen.csv not found.\n"
        f"Expected: {DEFAULT_GEN_FILE}\n"
        "Run `python scripts/prepare_external_data.py` or set "
        "FAME_RTS_GEN_DATA.\n"
        "See data/SOURCES_AND_RECONSTRUCTION.md."
    )

gen=pd.read_csv(GEN_FILE)

thermal_mask=~gen["Fuel"].astype(str).isin(
    ["Wind","Solar","Storage","Hydro","CSP"]
)

generators=gen.loc[thermal_mask].copy()

generators["PMax MW"]=pd.to_numeric(
    generators["PMax MW"],
    errors="coerce"
)

generators=generators[
    generators["PMax MW"].notna()
    & (generators["PMax MW"]>0)
].reset_index(drop=True)

def representative_cost(row):
    pmax=float(row["PMax MW"])

    fuel=pd.to_numeric(
        pd.Series([
            row.get("Fuel Price $/MMBTU",np.nan)
        ]),
        errors="coerce"
    ).iloc[0]

    op=pd.to_numeric(
        pd.Series([
            row.get("Output_pct_3",np.nan)
        ]),
        errors="coerce"
    ).iloc[0]

    hr=pd.to_numeric(
        pd.Series([
            row.get("HR_incr_3",np.nan)
        ]),
        errors="coerce"
    ).iloc[0]

    fuel=1.0 if pd.isna(fuel) or fuel<=0 else float(fuel)
    op=1.0 if pd.isna(op) or op<=0 else float(op)
    hr=10_000.0 if pd.isna(hr) or hr<=0 else float(hr)

    return max(
        pmax*op*hr*fuel/1000.0,
        1e-6
    )

generators["capacity_mw"]=generators[
    "PMax MW"
].astype(float)

generators["decision_cost"]=generators.apply(
    representative_cost,
    axis=1
)

generators["g"]=np.arange(
    len(generators),
    dtype=int
)

generator_table=generators[
    ["g","Fuel","capacity_mw","decision_cost"]
].copy()

generator_table["cost_per_mw"]=(
    generator_table["decision_cost"]
    /
    generator_table["capacity_mw"].clip(lower=1e-9)
)

TOTAL_CONVENTIONAL_CAPACITY=float(
    generator_table["capacity_mw"].sum()
)

EMERGENCY_COST_PER_MW=float(
    EMERGENCY_COST_MULTIPLIER
    *
    generator_table["cost_per_mw"].max()
)

print("Thermal capacity:",TOTAL_CONVENTIONAL_CAPACITY)
print("Emergency cost/MW:",EMERGENCY_COST_PER_MW)


In [ ]:
capacities = generator_table["capacity_mw"].to_numpy(dtype=float)
costs = generator_table["decision_cost"].to_numpy(dtype=float)

def infer_capacity_scale(values, max_decimals=3, tol=1e-8):
    for decimals in range(max_decimals + 1):
        scale = 10 ** decimals
        if np.allclose(values * scale, np.round(values * scale), atol=tol, rtol=0):
            return scale, decimals
    return None, None

CAPACITY_SCALE, CAPACITY_DECIMALS = infer_capacity_scale(capacities)
USE_FAST_DP = CAPACITY_SCALE is not None

if USE_FAST_DP:
    cap_units = np.rint(capacities * CAPACITY_SCALE).astype(int)
    total_units = int(cap_units.sum())
    if total_units > 500_000:
        print("Capacity grid too large for fast DP; using cached CBC fallback.")
        USE_FAST_DP = False

if USE_FAST_DP:
    INF = np.inf
    dp_cost = np.full(total_units + 1, INF, dtype=float)
    dp_count = np.full(total_units + 1, np.iinfo(np.int32).max, dtype=np.int32)
    dp_cost[0] = 0.0
    dp_count[0] = 0

    for q, c in zip(cap_units, costs):
        old_cost = dp_cost.copy()
        old_count = dp_count.copy()
        src = np.arange(0, total_units - q + 1)
        feasible = np.isfinite(old_cost[src])
        src = src[feasible]
        dst_idx = src + q
        cand_cost = old_cost[src] + c
        cand_count = old_count[src] + 1
        better = cand_cost < dp_cost[dst_idx] - 1e-10
        tie = np.isclose(cand_cost, dp_cost[dst_idx], rtol=1e-12, atol=1e-10) & (cand_count < dp_count[dst_idx])
        update = better | tie
        dp_cost[dst_idx[update]] = cand_cost[update]
        dp_count[dst_idx[update]] = cand_count[update]

    suffix_cost = np.full(total_units + 1, INF, dtype=float)
    suffix_count = np.full(total_units + 1, np.iinfo(np.int32).max, dtype=np.int32)
    suffix_capacity_units = np.full(total_units + 1, -1, dtype=np.int32)
    best_cost = INF
    best_count = np.iinfo(np.int32).max
    best_q = -1
    for q in range(total_units, -1, -1):
        c = dp_cost[q]
        n = dp_count[q]
        if c < best_cost - 1e-10 or (np.isclose(c, best_cost, rtol=1e-12, atol=1e-10) and n < best_count):
            best_cost, best_count, best_q = c, n, q
        suffix_cost[q] = best_cost
        suffix_count[q] = best_count
        suffix_capacity_units[q] = best_q

    print("Fast exact DP enabled")
    print("Capacity decimals:", CAPACITY_DECIMALS)
    print("DP states:", total_units + 1)
else:
    print("Fast DP unavailable; cached CBC fallback will be used")

CBC_PLAN_CACHE = {}


In [ ]:
def _solve_capacity_plan_cbc(required_capacity_mw):
    required_capacity_mw = float(required_capacity_mw)
    cache_key = round(required_capacity_mw, 9)
    if cache_key in CBC_PLAN_CACHE:
        return CBC_PLAN_CACHE[cache_key]

    emergency_upper = max(0.0, required_capacity_mw - TOTAL_CONVENTIONAL_CAPACITY)
    problem = pulp.LpProblem("FAME_Energy_Capacity", pulp.LpMinimize)
    x = {
        int(r.g): pulp.LpVariable(f"x_{int(r.g)}", 0, 1, cat=pulp.LpBinary)
        for r in generator_table.itertuples(index=False)
    }
    emergency = pulp.LpVariable("emergency_mw", lowBound=0, upBound=emergency_upper, cat=pulp.LpContinuous)
    problem += (
        pulp.lpSum(float(r.decision_cost) * x[int(r.g)] for r in generator_table.itertuples(index=False))
        + EMERGENCY_COST_PER_MW * emergency
    )
    problem += (
        pulp.lpSum(float(r.capacity_mw) * x[int(r.g)] for r in generator_table.itertuples(index=False))
        + emergency >= required_capacity_mw
    )
    solver = pulp.COIN_CMD(path=str(CBC_EXE), msg=False) if CBC_EXE.is_file() else pulp.PULP_CBC_CMD(msg=False)
    status = problem.solve(solver)
    if pulp.LpStatus[status] != "Optimal":
        CBC_PLAN_CACHE[cache_key] = None
        return None
    selected = [
        int(r.g) for r in generator_table.itertuples(index=False)
        if x[int(r.g)].value() is not None and x[int(r.g)].value() > 0.5
    ]
    sel = generator_table[generator_table["g"].isin(selected)]
    emergency_mw = float(emergency.value() or 0.0)
    conventional_capacity = float(sel["capacity_mw"].sum())
    conventional_cost = float(sel["decision_cost"].sum())
    ans = {
        "available_capacity_mw": conventional_capacity + emergency_mw,
        "emergency_capacity_mw": emergency_mw,
        "decision_cost": conventional_cost + EMERGENCY_COST_PER_MW * emergency_mw,
        "n_generators": len(selected),
    }
    CBC_PLAN_CACHE[cache_key] = ans
    return ans

def solve_capacity_plan(required_capacity_mw):
    required_capacity_mw = float(required_capacity_mw)
    if not USE_FAST_DP:
        return _solve_capacity_plan_cbc(required_capacity_mw)

    if required_capacity_mw <= TOTAL_CONVENTIONAL_CAPACITY + 1e-10:
        required_units = int(np.ceil(required_capacity_mw * CAPACITY_SCALE - 1e-10))
        required_units = max(0, min(required_units, total_units))
        q = int(suffix_capacity_units[required_units])
        if q < 0 or not np.isfinite(suffix_cost[required_units]):
            return _solve_capacity_plan_cbc(required_capacity_mw)
        return {
            "available_capacity_mw": float(q / CAPACITY_SCALE),
            "emergency_capacity_mw": 0.0,
            "decision_cost": float(suffix_cost[required_units]),
            "n_generators": int(suffix_count[required_units]),
        }

    emergency_mw = required_capacity_mw - TOTAL_CONVENTIONAL_CAPACITY
    return {
        "available_capacity_mw": float(required_capacity_mw),
        "emergency_capacity_mw": float(emergency_mw),
        "decision_cost": float(generator_table["decision_cost"].sum() + EMERGENCY_COST_PER_MW * emergency_mw),
        "n_generators": int(len(generator_table)),
    }


In [ ]:
def evaluate_decision(
    date,
    model,
    forecast_peak,
    actual_peak,
    theta,
    voll
):
    """
    Avalia uma decisão FAME usando exatamente o mesmo cálculo de perda
    da v3.3 original.
    """
    required = float(forecast_peak) * np.exp(float(theta))

    plan = solve_capacity_plan(required)

    if plan is None:
        return {
            "date": date,
            "model": model,
            "theta": float(theta),
            "voll": float(voll),
            "status": "solver_failure",
            "realized_loss": np.inf
        }

    shortage = max(
        0.0,
        float(actual_peak)
        - plan["available_capacity_mw"]
    )

    return {
        "date": date,
        "model": model,
        "theta": float(theta),
        "voll": float(voll),
        "forecast_peak": float(forecast_peak),
        "actual_peak": float(actual_peak),
        "required_capacity_mw": required,
        "available_capacity_mw":
            plan["available_capacity_mw"],
        "emergency_capacity_mw":
            plan["emergency_capacity_mw"],
        "decision_cost":
            plan["decision_cost"],
        "n_generators":
            plan["n_generators"],
        "shortage_mw":
            shortage,
        "realized_loss":
            plan["decision_cost"]
            + float(voll) * shortage,
        "status": "optimal"
    }

print("evaluate_decision definida: OK")


In [ ]:
if USE_FAST_DP:
    rng = np.random.default_rng(SEED)
    audit_requirements = np.concatenate([
        np.array([
            0.10 * TOTAL_CONVENTIONAL_CAPACITY,
            0.50 * TOTAL_CONVENTIONAL_CAPACITY,
            0.90 * TOTAL_CONVENTIONAL_CAPACITY,
            0.999 * TOTAL_CONVENTIONAL_CAPACITY,
            1.01 * TOTAL_CONVENTIONAL_CAPACITY,
        ]),
        rng.uniform(0.05 * TOTAL_CONVENTIONAL_CAPACITY, 1.05 * TOTAL_CONVENTIONAL_CAPACITY, size=10)
    ])
    audit_rows=[]
    for req in audit_requirements:
        fast=solve_capacity_plan(float(req))
        cbc=_solve_capacity_plan_cbc(float(req))
        if fast is None or cbc is None:
            raise RuntimeError(f"Audit solve failed at {req}")
        audit_rows.append({
            "required_capacity_mw":req,
            "fast_cost":fast["decision_cost"],
            "cbc_cost":cbc["decision_cost"],
            "cost_difference":fast["decision_cost"]-cbc["decision_cost"],
        })
        if not np.isclose(fast["decision_cost"],cbc["decision_cost"],rtol=1e-9,atol=1e-6):
            raise RuntimeError(f"DP cost differs from CBC at {req:.6f} MW")
    dp_cbc_audit=pd.DataFrame(audit_rows)
    display(dp_cbc_audit)
    print("DP versus CBC audit: OK")
else:
    print("DP audit skipped; cached CBC fallback active")


In [ ]:
prediction_cache={}
predictive_rows=[]

for rep in replications.itertuples(index=False):
    rid=rep.replication_id
    print(rid)

    dev=daily[
        daily.year.between(
            rep.development_start,
            rep.development_end
        )
    ].copy()

    cal=daily[
        daily.year.eq(rep.calibration_year)
    ].copy()

    tst=daily[
        daily.year.eq(rep.test_year)
    ].copy()

    sar_order,sar_seasonal,_=select_sarimax(dev)

    lstm_cfg,_=select_lstm(dev)

    lstm_model,lstm_scaler=fit_lstm_final(
        dev,
        lstm_cfg
    )

    sar_cal=sarimax_sequential_forecast(
        dev,
        cal,
        sar_order,
        sar_seasonal
    )

    lstm_cal=lstm_sequential_forecast(
        lstm_model,
        lstm_scaler,
        dev,
        cal,
        lstm_cfg["lookback"]
    )

    dev_cal=pd.concat(
        [dev,cal],
        ignore_index=True
    ).sort_values("date")

    sar_test=sarimax_sequential_forecast(
        dev_cal,
        tst,
        sar_order,
        sar_seasonal
    )

    lstm_test=lstm_sequential_forecast(
        lstm_model,
        lstm_scaler,
        dev_cal,
        tst,
        lstm_cfg["lookback"]
    )

    cal_pred=cal[
        ["date","actual_peak","entsoe_forecast_peak"]
    ].copy()

    cal_pred=cal_pred.rename(
        columns={
            "entsoe_forecast_peak":"ENTSOE"
        }
    )

    cal_pred["SARIMAX"]=sar_cal
    cal_pred["LSTM"]=lstm_cal

    tst_pred=tst[
        ["date","actual_peak","entsoe_forecast_peak"]
    ].copy()

    tst_pred=tst_pred.rename(
        columns={
            "entsoe_forecast_peak":"ENTSOE"
        }
    )

    tst_pred["SARIMAX"]=sar_test
    tst_pred["LSTM"]=lstm_test

    prediction_cache[rid]={
        "dev_mean_actual":
            float(dev["actual_peak"].mean()),
        "cal":cal_pred,
        "test":tst_pred,
        "sarimax_order":sar_order,
        "sarimax_seasonal":sar_seasonal,
        "lstm_cfg":lstm_cfg
    }

    for stage,df in [
        ("calibration",cal_pred),
        ("test",tst_pred)
    ]:
        for model in [
            "ENTSOE",
            "SARIMAX",
            "LSTM"
        ]:
            predictive_rows.append({
                "replication_id":rid,
                "stage":stage,
                "model":model,
                **regression_metrics(
                    df["actual_peak"],
                    df[model]
                )
            })

predictive=pd.DataFrame(predictive_rows)

predictive.to_csv(
    OUT/"predictive_metrics_native_scale.csv",
    index=False
)

print("Prediction cache complete.")


In [ ]:
required_objects = [
    "replications",
    "prediction_cache",
    "CAPACITY_MARGINS",
    "VOLL_MULTIPLIERS",
    "THETA_GRID",
    "TOTAL_CONVENTIONAL_CAPACITY",
    "solve_capacity_plan",
    "evaluate_decision",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Dependências ausentes antes do Bloco 8: "
        + ", ".join(missing_objects)
    )

_audit_req = 0.50 * TOTAL_CONVENTIONAL_CAPACITY
_audit_plan = solve_capacity_plan(_audit_req)

if _audit_plan is None:
    raise RuntimeError("Sanity check do solver falhou.")

_audit_eval = evaluate_decision(
    date=pd.Timestamp("2000-01-01"),
    model="AUDIT",
    forecast_peak=_audit_req,
    actual_peak=_audit_req,
    theta=0.0,
    voll=BASE_VOLL
)

if not np.isfinite(_audit_eval["realized_loss"]):
    raise RuntimeError("Sanity check de evaluate_decision falhou.")

print("Auditoria pré-Bloco 8: OK")
print("Solver mode:", "Fast DP" if USE_FAST_DP else "CBC fallback")


In [ ]:
import time

block8_t0=time.perf_counter()
cal_surface_rows=[]
test_rows=[]
freeze_rows=[]

THETA_MIN=float(THETA_GRID.min())
THETA_MAX=float(THETA_GRID.max())

for rep in replications.itertuples(index=False):
    rid=rep.replication_id
    cache=prediction_cache[rid]

    print("\n",rid)

    for margin in CAPACITY_MARGINS:
        scale=(
            TOTAL_CONVENTIONAL_CAPACITY
            /
            (
                float(margin)
                * cache["dev_mean_actual"]
            )
        )

        cal=cache["cal"].copy()
        tst=cache["test"].copy()

        for c in [
            "actual_peak",
            "ENTSOE",
            "SARIMAX",
            "LSTM"
        ]:
            cal[c]=cal[c]*scale
            tst[c]=tst[c]*scale

        for voll_mult in VOLL_MULTIPLIERS:
            voll=float(
                BASE_VOLL*voll_mult
            )

            theta_map={}

            for model in [
                "ENTSOE",
                "SARIMAX",
                "LSTM"
            ]:
                model_surface=[]

                for theta in THETA_GRID:
                    rows=[
                        evaluate_decision(
                            row.date,
                            model,
                            getattr(row,model),
                            row.actual_peak,
                            float(theta),
                            voll
                        )
                        for row in cal.itertuples(index=False)
                    ]

                    df=pd.DataFrame(rows)

                    if not np.isfinite(
                        df["realized_loss"]
                    ).all():
                        raise RuntimeError(
                            f"{rid}, M={margin}, "
                            f"VOLLx={voll_mult}, "
                            f"{model}: non-finite loss."
                        )

                    rec={
                        "replication_id":rid,
                        "margin":float(margin),
                        "voll_multiplier":float(voll_mult),
                        "voll":voll,
                        "model":model,
                        "theta":float(theta),
                        "mean_loss":
                            float(df["realized_loss"].mean()),
                        "total_shortage_mw":
                            float(df["shortage_mw"].sum()),
                        "total_emergency_mw":
                            float(df["emergency_capacity_mw"].sum()),
                        "mean_decision_cost":
                            float(df["decision_cost"].mean())
                    }

                    cal_surface_rows.append(rec)
                    model_surface.append(rec)

                surf=pd.DataFrame(model_surface)

                best_loss=surf["mean_loss"].min()

                cand=surf[
                    np.isclose(
                        surf["mean_loss"],
                        best_loss,
                        rtol=1e-10,
                        atol=1e-8
                    )
                ].copy()

                cand["abs_theta"]=cand[
                    "theta"
                ].abs()

                best=cand.sort_values(
                    ["abs_theta","theta"]
                ).iloc[0]

                theta_doc=float(best["theta"])
                theta_map[model]=theta_doc

                boundary=bool(
                    np.isclose(theta_doc,THETA_MIN)
                    or
                    np.isclose(theta_doc,THETA_MAX)
                )

                freeze_rows.append({
                    "replication_id":rid,
                    "margin":float(margin),
                    "voll_multiplier":float(voll_mult),
                    "model":model,
                    "theta_doc":theta_doc,
                    "boundary_selected":boundary,
                    "scale_factor":float(scale)
                })

                for theta,strategy in [
                    (0.0,"baseline"),
                    (theta_doc,"FAME-DOC")
                ]:
                    for row in tst.itertuples(index=False):
                        rr=evaluate_decision(
                            row.date,
                            model,
                            getattr(row,model),
                            row.actual_peak,
                            float(theta),
                            voll
                        )

                        rr["replication_id"]=rid
                        rr["margin"]=float(margin)
                        rr["voll_multiplier"]=float(voll_mult)
                        rr["strategy"]=strategy

                        test_rows.append(rr)

    elapsed=time.perf_counter()-block8_t0
    completed_rep=int(str(rid).replace("E",""))-2016
    avg_per_rep=elapsed/completed_rep
    eta=avg_per_rep*(len(replications)-completed_rep)
    print(rid,"completed | elapsed",f"{elapsed/60:.2f} min | ETA",f"{eta/60:.2f} min")

cal_surface=pd.DataFrame(cal_surface_rows)
test_decisions=pd.DataFrame(test_rows)
freezes=pd.DataFrame(freeze_rows)

cal_surface.to_csv(
    OUT/"calibration_surface_margin_voll.csv",
    index=False
)

test_decisions.to_csv(
    OUT/"test_decisions_margin_voll.csv",
    index=False
)

freezes.to_csv(
    OUT/"theta_freezes_margin_voll.csv",
    index=False
)

block8_elapsed=time.perf_counter()-block8_t0
print("\nBlock 8 completed in",f"{block8_elapsed/60:.2f} minutes")


In [ ]:
boundary_summary=(
    freezes.groupby(
        ["margin","voll_multiplier","model"],
        as_index=False
    )
    .agg(
        n_replications=("replication_id","count"),
        boundary_rate=("boundary_selected","mean"),
        mean_theta=("theta_doc","mean"),
        sd_theta=("theta_doc","std"),
        min_theta=("theta_doc","min"),
        max_theta=("theta_doc","max"),
    )
)

boundary_summary["boundary_rate_pct"] = (
    100*boundary_summary["boundary_rate"]
)

display(boundary_summary)

boundary_summary.to_csv(
    OUT/"boundary_selection_summary.csv",
    index=False
)


In [ ]:
test_summary=(
    test_decisions.groupby(
        [
            "replication_id",
            "margin",
            "voll_multiplier",
            "model",
            "strategy"
        ],
        as_index=False
    )
    .agg(
        mean_loss=("realized_loss","mean"),
        total_shortage_mw=("shortage_mw","sum"),
        total_emergency_mw=("emergency_capacity_mw","sum")
    )
)

wide=test_summary.pivot(
    index=[
        "replication_id",
        "margin",
        "voll_multiplier",
        "model"
    ],
    columns="strategy",
    values="mean_loss"
).reset_index()

wide["gain"]=(
    wide["baseline"]
    -
    wide["FAME-DOC"]
)

wide["gain_pct"]=(
    100
    * wide["gain"]
    / wide["baseline"]
)

wide["transferred"]=(
    wide["gain"]>0
)

robustness=(
    wide.groupby(
        [
            "margin",
            "voll_multiplier",
            "model"
        ],
        as_index=False
    )
    .agg(
        n_replications=("replication_id","count"),
        transfer_rate=("transferred","mean"),
        mean_gain_pct=("gain_pct","mean"),
        median_gain_pct=("gain_pct","median"),
        min_gain_pct=("gain_pct","min"),
        max_gain_pct=("gain_pct","max")
    )
)

robustness["transfer_rate_pct"] = (
    100*robustness["transfer_rate"]
)

display(robustness)

robustness.to_csv(
    OUT/"robustness_margin_voll_summary.csv",
    index=False
)


In [ ]:
global_summary=(
    wide.groupby("model",as_index=False)
    .agg(
        n_scenarios=("transferred","count"),
        transfer_rate=("transferred","mean"),
        mean_gain_pct=("gain_pct","mean"),
        median_gain_pct=("gain_pct","median"),
        min_gain_pct=("gain_pct","min"),
        max_gain_pct=("gain_pct","max")
    )
)

global_summary["transfer_rate_pct"] = (
    100*global_summary["transfer_rate"]
)

display(global_summary)

global_summary.to_csv(
    OUT/"global_robustness_summary_by_model.csv",
    index=False
)


In [ ]:
for model in ["ENTSOE","SARIMAX","LSTM"]:
    p=robustness[
        robustness["model"]==model
    ].pivot(
        index="voll_multiplier",
        columns="margin",
        values="transfer_rate_pct"
    )

    plt.figure(figsize=(8,4.5))
    plt.imshow(
        p.values,
        aspect="auto",
        origin="lower"
    )

    plt.xticks(
        range(len(p.columns)),
        [f"{x:.2f}" for x in p.columns]
    )

    plt.yticks(
        range(len(p.index)),
        [f"{x:.1f}" for x in p.index]
    )

    plt.xlabel("Capacity margin")
    plt.ylabel("VOLL multiplier")
    plt.title(
        f"Transfer rate (%) — {model}"
    )

    for i in range(p.shape[0]):
        for j in range(p.shape[1]):
            plt.text(
                j,i,
                f"{p.iloc[i,j]:.0f}",
                ha="center",
                va="center"
            )

    plt.colorbar(label="Transfer rate (%)")
    plt.show()


In [ ]:
for model in ["ENTSOE","SARIMAX","LSTM"]:
    p=boundary_summary[
        boundary_summary["model"]==model
    ].pivot(
        index="voll_multiplier",
        columns="margin",
        values="boundary_rate_pct"
    )

    plt.figure(figsize=(8,4.5))
    plt.imshow(
        p.values,
        aspect="auto",
        origin="lower"
    )

    plt.xticks(
        range(len(p.columns)),
        [f"{x:.2f}" for x in p.columns]
    )

    plt.yticks(
        range(len(p.index)),
        [f"{x:.1f}" for x in p.index]
    )

    plt.xlabel("Capacity margin")
    plt.ylabel("VOLL multiplier")
    plt.title(
        f"Boundary-selection rate (%) — {model}"
    )

    for i in range(p.shape[0]):
        for j in range(p.shape[1]):
            plt.text(
                j,i,
                f"{p.iloc[i,j]:.0f}",
                ha="center",
                va="center"
            )

    plt.colorbar(label="Boundary rate (%)")
    plt.show()
